# Topic 12: Greedy Algorithms

**Goal**: Understand when greedy works, why it works, and solve the classic greedy problems.  
**Time**: ~4-5 hours  
**Prereqs**: Topics 7-8

---

## What Is Greedy?

At each step, make the **locally optimal** choice, hoping it leads to a **globally optimal** solution.  
Unlike DP, you never look back or reconsider.

```
Greedy vs DP:

  Greedy: Make the best choice NOW  →  hope it works globally
  DP:     Try ALL choices           →  pick the best combination

  ┌──────────────────────────────────────────────────┐
  │  Greedy is faster but doesn't always give the    │
  │  optimal answer. The hard part: PROVING that     │
  │  greedy gives the correct answer.                │
  └──────────────────────────────────────────────────┘
```

## When Does Greedy Work?

Two conditions must hold:

1. **Greedy choice property** — a locally optimal choice leads to a globally optimal solution  
2. **Optimal substructure** — optimal solution contains optimal sub-solutions

```
  GREEDY DECISION FLOW:

  Problem → Can I sort + pick the "best" at each step?
                │                         │
               YES                       NO
                │                         │
         Can I prove it?          Try DP / backtracking
                │
          YES → GREEDY!
```

---

## Problem 1: Activity Selection / Interval Scheduling

**Problem**: Given start/end times, select the **maximum** number of non-overlapping activities.

**Greedy Strategy**: Always pick the activity that **ends earliest**.

```
Activities: [(1,4), (3,5), (0,6), (5,7), (3,8), (5,9), (6,10), (8,11), (8,12), (2,13)]

Sort by end time → pick greedily:

  Time:  0  1  2  3  4  5  6  7  8  9  10 11 12 13
         ├──┼──┼──┼──┼──┼──┼──┼──┼──┼──┼──┼──┼──┤
  (1,4)  ·  ████████·  ·  ·  ·  ·  ·  ·  ·  ·  ·   ✓ PICK (ends earliest)
  (3,5)  ·  ·  ·  ██████  ·  ·  ·  ·  ·  ·  ·  ·   ✗ overlaps (1,4)
  (0,6)  ████████████████  ·  ·  ·  ·  ·  ·  ·  ·   ✗ overlaps (1,4)
  (5,7)  ·  ·  ·  ·  ·  ██████  ·  ·  ·  ·  ·  ·   ✓ PICK (start ≥ 4)
  (3,8)  ·  ·  ·  ████████████  ·  ·  ·  ·  ·  ·   ✗ overlaps (5,7)
  (5,9)  ·  ·  ·  ·  ·  ████████████  ·  ·  ·  ·   ✗ overlaps (5,7)
  (6,10) ·  ·  ·  ·  ·  ·  ████████████  ·  ·  ·   ✗ overlaps (5,7)
  (8,11) ·  ·  ·  ·  ·  ·  ·  ·  ████████████  ·   ✓ PICK (start ≥ 7)
  (8,12) ·  ·  ·  ·  ·  ·  ·  ·  ████████████████   ✗ overlaps (8,11)
  (2,13) ·  ·  ██████████████████████████████████·   ✗ overlaps (8,11)

  Result: 3 activities selected — this IS optimal!
```

In [ ]:
def activity_selection(activities):
    sorted_acts = sorted(activities, key=lambda x: x[1])
    print(f"Sorted by end time: {sorted_acts}\n")

    selected = [sorted_acts[0]]
    print(f"  Pick {sorted_acts[0]}  (first activity)")

    for i in range(1, len(sorted_acts)):
        start, end = sorted_acts[i]
        last_end = selected[-1][1]
        if start >= last_end:
            selected.append(sorted_acts[i])
            print(f"  Pick {sorted_acts[i]}  (start {start} >= last_end {last_end})")
        else:
            print(f"  Skip {sorted_acts[i]}  (start {start} < last_end {last_end})")

    return selected


activities = [(1,4), (3,5), (0,6), (5,7), (3,8), (5,9), (6,10), (8,11), (8,12), (2,13)]
result = activity_selection(activities)
print(f"\nMax non-overlapping: {len(result)} → {result}")

### Why Earliest-End Works — Proof Sketch

```
  Proof by exchange argument:

  Suppose OPT is an optimal solution, and GREEDY picks a different
  first activity. GREEDY picks the one that ends earliest.

  OPT:     [──A──]  [──B──]  [──C──]  ...   (some first activity A)
  GREEDY:  [─G─]    [──B──]  [──C──]  ...   (G ends ≤ A)

  Since G ends no later than A, we can SWAP G for A in OPT
  without breaking any constraints. The rest still fits!

  → GREEDY's choice is at least as good as OPT's at every step
  → GREEDY produces an optimal solution  ∎
```

---

## Problem 2: Fractional Knapsack

Unlike 0/1 knapsack (which requires DP), here you can take **fractions** of items.

**Greedy**: Sort by **value/weight ratio**, take as much as possible of the best ratio first.

```
  0/1 Knapsack:          Fractional Knapsack:
  ┌──────────┐           ┌──────────┐
  │ Take it  │           │ Take ALL │
  │   or     │    vs     │  or take │
  │ leave it │           │ a PIECE  │
  └──────────┘           └──────────┘
     → DP                   → GREEDY

  Items: [(weight=10, value=60), (weight=20, value=100), (weight=30, value=120)]
  Capacity: 50

  Ratios:  60/10 = 6.0   100/20 = 5.0   120/30 = 4.0
  Sort:    Item1(6.0) → Item2(5.0) → Item3(4.0)

  Take Item1 fully:  10kg used,  value = 60,   remaining = 40
  Take Item2 fully:  20kg used,  value = 100,  remaining = 20
  Take 2/3 of Item3: 20kg used,  value = 80,   remaining = 0
                                 ─────────────
  Total value = 240
```

In [ ]:
def fractional_knapsack(items, capacity):
    for i, (w, v) in enumerate(items):
        items[i] = (w, v, v / w)
    items.sort(key=lambda x: x[2], reverse=True)

    print(f"Sorted by value/weight ratio:")
    for w, v, r in items:
        print(f"  weight={w}, value={v}, ratio={r:.2f}")
    print(f"\nCapacity: {capacity}\n")

    total_value = 0
    remaining = capacity

    for weight, value, ratio in items:
        if remaining <= 0:
            break
        take = min(weight, remaining)
        fraction = take / weight
        gained = value * fraction
        total_value += gained
        remaining -= take
        status = "FULL" if fraction == 1.0 else f"{fraction:.2f}"
        print(f"  Take {status} of (w={weight}, v={value}) → +{gained:.0f} value | remaining capacity: {remaining}")

    return total_value


items = [(10, 60), (20, 100), (30, 120)]
result = fractional_knapsack(items, 50)
print(f"\nMax value: {result}")

---

## Problem 3: Jump Game I — LC #55

**Problem**: Given array where `nums[i]` = max jump length from position `i`, can you reach the last index?

**Greedy**: Track the **farthest** index reachable. If current index exceeds farthest, you're stuck.

```
  nums = [2, 3, 1, 1, 4]

  i=0: farthest = max(0, 0+2) = 2    can reach index 2
  i=1: farthest = max(2, 1+3) = 4    can reach index 4  ← that's the end!
  → TRUE

  nums = [3, 2, 1, 0, 4]

  i=0: farthest = max(0, 0+3) = 3
  i=1: farthest = max(3, 1+2) = 3
  i=2: farthest = max(3, 2+1) = 3
  i=3: farthest = max(3, 3+0) = 3    stuck at 3, can't reach 4
  → FALSE
```

In [ ]:
def can_jump(nums):
    farthest = 0
    print(f"nums = {nums}  (need to reach index {len(nums) - 1})\n")

    for i in range(len(nums)):
        if i > farthest:
            print(f"  i={i}: STUCK — can't reach this index (farthest={farthest})")
            return False
        farthest = max(farthest, i + nums[i])
        print(f"  i={i}: nums[{i}]={nums[i]} → farthest = {farthest}")
        if farthest >= len(nums) - 1:
            print(f"  → Can reach the end!")
            return True

    return farthest >= len(nums) - 1


print("Test 1:"); print(f"Result: {can_jump([2, 3, 1, 1, 4])}\n")
print("Test 2:"); print(f"Result: {can_jump([3, 2, 1, 0, 4])}")

---

## Problem 4: Jump Game II — LC #45

**Problem**: Minimum number of jumps to reach the last index. (Guaranteed reachable.)

**Greedy (BFS-style)**: Think of it as levels in BFS. Each "level" is the range of indices
reachable with the current number of jumps.

```
  nums = [2, 3, 1, 1, 4]

  Level 0: [2]           → can reach indices 1..2
  Level 1: [3, 1]        → can reach indices 2..4
  Level 2: reached end!

  ┌───┬───┬───┬───┬───┐
  │ 2 │ 3 │ 1 │ 1 │ 4 │
  └───┴───┴───┴───┴───┘
   L0   ←L1→   ← L2 →
  jumps = 2
```

In [ ]:
def jump_game_ii(nums):
    if len(nums) <= 1:
        return 0

    jumps = 0
    current_end = 0
    farthest = 0
    print(f"nums = {nums}\n")

    for i in range(len(nums) - 1):
        farthest = max(farthest, i + nums[i])
        print(f"  i={i}: nums[{i}]={nums[i]} → farthest={farthest}", end="")

        if i == current_end:
            jumps += 1
            current_end = farthest
            print(f"  ← end of level → JUMP #{jumps} (new boundary: {current_end})")
            if current_end >= len(nums) - 1:
                print(f"  → Reached the end!")
                break
        else:
            print()

    return jumps


print(f"Min jumps: {jump_game_ii([2, 3, 1, 1, 4])}\n")
print(f"Min jumps: {jump_game_ii([2, 3, 0, 1, 4])}")

---

## Problem 5: Gas Station — LC #134

**Problem**: `n` gas stations in a circle. `gas[i]` = fuel gained, `cost[i]` = fuel to next station.  
Find the starting station to complete the circuit, or return `-1`.

**Key Insights**:
1. If `sum(gas) >= sum(cost)`, a solution **always** exists
2. If running surplus drops below 0 at station `i`, **none** of `0..i` can be the start → try `i+1`

```
  gas  = [1, 2, 3, 4, 5]
  cost = [3, 4, 5, 1, 2]
  net  = [-2,-2,-2, 3, 3]    (gas[i] - cost[i])

  total = sum(net) = 0 ≥ 0  →  solution exists!

  Scan for start:
  i=0: tank= -2 < 0 → reset, start=1
  i=1: tank= -2 < 0 → reset, start=2
  i=2: tank= -2 < 0 → reset, start=3
  i=3: tank=  3 ≥ 0 → keep going
  i=4: tank=  6 ≥ 0 → keep going
  Start at station 3  ✓
```

In [ ]:
def can_complete_circuit(gas, cost):
    total_surplus = 0
    current_surplus = 0
    start = 0

    print(f"gas  = {gas}")
    print(f"cost = {cost}")
    print(f"net  = {[g - c for g, c in zip(gas, cost)]}\n")

    for i in range(len(gas)):
        net = gas[i] - cost[i]
        total_surplus += net
        current_surplus += net

        if current_surplus < 0:
            print(f"  i={i}: net={net:+d}  tank={current_surplus:+d} < 0 → reset, start={i + 1}")
            start = i + 1
            current_surplus = 0
        else:
            print(f"  i={i}: net={net:+d}  tank={current_surplus:+d} ≥ 0 → keep going")

    if total_surplus < 0:
        print(f"\nTotal surplus = {total_surplus} < 0 → impossible")
        return -1

    print(f"\nTotal surplus = {total_surplus} ≥ 0 → start at station {start}")
    return start


print(f"Answer: {can_complete_circuit([1,2,3,4,5], [3,4,5,1,2])}\n")
print(f"Answer: {can_complete_circuit([2,3,4], [3,4,3])}")

---

## Problem 6: Candy Distribution — LC #135

**Problem**: Each child has a rating. Give candies such that:
- Every child gets at least 1 candy
- Higher-rated child gets more than their neighbor

Minimize total candies.

**Greedy (Two Passes)**:
- **Left → Right**: if `rating[i] > rating[i-1]`, give `candy[i] = candy[i-1] + 1`
- **Right → Left**: if `rating[i] > rating[i+1]`, ensure `candy[i] >= candy[i+1] + 1`

```
  ratings:   [1,  0,  2]

  Pass 1 (L→R):  [1,  1,  2]   (0 < 1 so no bump; 2 > 0 so bump)
  Pass 2 (R→L):  [2,  1,  2]   (1 > 0 so candy[0] = max(1, 1+1) = 2)
  Total: 5
```

In [ ]:
def candy(ratings):
    n = len(ratings)
    candies = [1] * n
    print(f"ratings: {ratings}")
    print(f"initial: {candies}\n")

    for i in range(1, n):
        if ratings[i] > ratings[i - 1]:
            candies[i] = candies[i - 1] + 1
    print(f"After L→R pass: {candies}")

    for i in range(n - 2, -1, -1):
        if ratings[i] > ratings[i + 1]:
            candies[i] = max(candies[i], candies[i + 1] + 1)
    print(f"After R←L pass: {candies}")

    total = sum(candies)
    print(f"\nDistribution: {list(zip(ratings, candies))}")
    print(f"Total candies: {total}")
    return total


candy([1, 0, 2])
print()
candy([1, 2, 2])
print()
candy([1, 3, 4, 5, 2])

---

## Problem 7: Non-overlapping Intervals — LC #435

**Problem**: Given intervals, find the **minimum** number of intervals to remove so the rest don't overlap.

**Greedy**: This is activity selection in disguise!  
Sort by end → greedily keep non-overlapping → count the ones you skip.

```
  intervals: [[1,2], [2,3], [3,4], [1,3]]
  sorted:    [[1,2], [2,3], [1,3], [3,4]]

  Keep [1,2] → Keep [2,3] → Skip [1,3] (overlaps) → Keep [3,4]
  Removed: 1
```

In [ ]:
def erase_overlap_intervals(intervals):
    intervals.sort(key=lambda x: x[1])
    print(f"Sorted by end: {intervals}\n")

    removals = 0
    prev_end = intervals[0][1]
    print(f"  Keep {intervals[0]}")

    for i in range(1, len(intervals)):
        start, end = intervals[i]
        if start < prev_end:
            removals += 1
            print(f"  REMOVE {intervals[i]}  (start {start} < prev_end {prev_end})")
        else:
            prev_end = end
            print(f"  Keep   {intervals[i]}  (start {start} >= prev_end {prev_end})")

    return removals


print(f"Removals: {erase_overlap_intervals([[1,2],[2,3],[3,4],[1,3]])}\n")
print(f"Removals: {erase_overlap_intervals([[1,2],[1,2],[1,2]])}\n")
print(f"Removals: {erase_overlap_intervals([[1,100],[11,22],[1,11],[2,12]])}")

---

## Problem 8: Meeting Rooms / Minimum Platforms

**Problem**: Given meeting intervals, what is the **minimum** number of rooms (or platforms) needed?

**Greedy (Event Sweep)**: Sort **start** and **end** times separately, sweep through.

```
  Meetings: [(0,30), (5,10), (15,20)]

  starts:  [0,  5, 15]     ends: [10, 20, 30]
            s    s   s            e   e    e

  Timeline sweep:
  ────────────────────────────────────────
  time 0:  Meeting starts → rooms=1
  time 5:  Meeting starts → rooms=2  ← peak!
  time 10: Meeting ends   → rooms=1
  time 15: Meeting starts → rooms=2  ← peak again
  time 20: Meeting ends   → rooms=1
  time 30: Meeting ends   → rooms=0
  ────────────────────────────────────────
  Max rooms needed: 2
```

In [ ]:
def min_meeting_rooms(intervals):
    starts = sorted(s for s, e in intervals)
    ends = sorted(e for s, e in intervals)
    print(f"Meetings: {intervals}")
    print(f"starts:   {starts}")
    print(f"ends:     {ends}\n")

    rooms = 0
    max_rooms = 0
    s_ptr, e_ptr = 0, 0

    while s_ptr < len(starts):
        if starts[s_ptr] < ends[e_ptr]:
            rooms += 1
            max_rooms = max(max_rooms, rooms)
            print(f"  time {starts[s_ptr]:>3}: START → rooms = {rooms}{'  ← peak!' if rooms == max_rooms and rooms > 1 else ''}")
            s_ptr += 1
        else:
            rooms -= 1
            print(f"  time {ends[e_ptr]:>3}: END   → rooms = {rooms}")
            e_ptr += 1

    while e_ptr < len(ends):
        rooms -= 1
        print(f"  time {ends[e_ptr]:>3}: END   → rooms = {rooms}")
        e_ptr += 1

    print(f"\nMinimum rooms: {max_rooms}")
    return max_rooms


min_meeting_rooms([(0, 30), (5, 10), (15, 20)])
print()
min_meeting_rooms([(1, 5), (2, 6), (3, 7), (4, 8)])
print()
min_meeting_rooms([(7, 10), (2, 4)])

---

## Problem 9: Assign Cookies — LC #455

**Problem**: Children have greed factors `g[i]`, cookies have sizes `s[j]`.  
A child is content if `s[j] >= g[i]`. Maximize content children.

**Greedy**: Sort both. Match the smallest satisfying cookie to the least greedy child.

```
  greed:   [1, 2, 3]      cookies: [1, 1]
  sorted:  [1, 2, 3]      sorted:  [1, 1]

  cookie 1 → child 1 (1 >= 1) ✓
  cookie 1 → child 2 (1 < 2)  ✗ no more cookies
  Content children: 1
```

In [ ]:
def find_content_children(g, s):
    g.sort()
    s.sort()
    print(f"Greed factors: {g}")
    print(f"Cookie sizes:  {s}\n")

    child = 0
    cookie = 0

    while child < len(g) and cookie < len(s):
        if s[cookie] >= g[child]:
            print(f"  Cookie {s[cookie]} → Child {child} (greed {g[child]}) ✓")
            child += 1
        else:
            print(f"  Cookie {s[cookie]} too small for Child {child} (greed {g[child]}) → skip cookie")
        cookie += 1

    print(f"\nContent children: {child}")
    return child


find_content_children([1, 2, 3], [1, 1])
print()
find_content_children([1, 2], [1, 2, 3])
print()
find_content_children([10, 9, 8, 7], [5, 6, 7, 8])

---

## Problem 10: Task Scheduler — LC #621

**Problem**: Given tasks with a cooldown `n`, find the minimum time to finish all tasks.  
Same tasks must have at least `n` intervals between them.

**Key Insight**: The most frequent task determines the "frame".

```
  tasks = [A, A, A, B, B, B]     n = 2

  Most frequent: A (count=3), B (count=3)  →  2 tasks with max freq

  Frame built around max frequency:
  A _ _ | A _ _ | A
  ╰─n─╯  ╰─n─╯

  Fill gaps with other tasks:
  A B _ | A B _ | A B

  Idle slots left: 2
  Total time = 8

  Formula:
  (max_freq - 1) × (n + 1) + count_of_max_freq_tasks
  = (3 - 1) × (2 + 1) + 2 = 8

  But if there are many tasks, there may be NO idle time:
  Answer = max(formula, len(tasks))
```

In [ ]:
from collections import Counter

def least_interval(tasks, n):
    freq = Counter(tasks)
    max_freq = max(freq.values())
    max_freq_count = sum(1 for v in freq.values() if v == max_freq)

    print(f"Tasks: {tasks}")
    print(f"Cooldown: n={n}")
    print(f"Frequencies: {dict(freq)}")
    print(f"Max frequency: {max_freq}  (shared by {max_freq_count} task(s))\n")

    frame_length = (max_freq - 1) * (n + 1) + max_freq_count
    result = max(frame_length, len(tasks))

    print(f"Frame: ({max_freq}-1) × ({n}+1) + {max_freq_count} = {frame_length}")
    print(f"Total tasks: {len(tasks)}")
    print(f"Answer: max({frame_length}, {len(tasks)}) = {result}")

    return result


least_interval(["A","A","A","B","B","B"], 2)
print()
least_interval(["A","A","A","B","B","B"], 0)
print()
least_interval(["A","A","A","A","A","A","B","C","D","E","F","G"], 2)

---

## Greedy vs DP — Decision Guide

```
┌────────────────────────────────────────────────────────────────────┐
│                    GREEDY vs DP DECISION GUIDE                    │
├────────────────────────────────────────────────────────────────────┤
│                                                                    │
│  Use GREEDY when:                                                  │
│    ✓ Sorting + picking the "best" option works                    │
│    ✓ You can prove locally optimal = globally optimal              │
│    ✓ Problem involves intervals, scheduling, or matching           │
│    ✓ Each choice is independent (doesn't affect future options)    │
│                                                                    │
│  Use DP when:                                                      │
│    ✗ Greedy doesn't give optimal (try a counterexample!)           │
│    ✗ Problem says "minimum/maximum" with constraints               │
│    ✗ Choices affect future choices (0/1 knapsack, coin change)     │
│    ✗ You need to try all combinations                              │
│                                                                    │
│  CLASSIC COMPARISON:                                               │
│  ┌──────────────────────┬──────────────────────┐                   │
│  │ Fractional Knapsack  │  0/1 Knapsack        │                   │
│  │ → GREEDY (can split) │  → DP (take or skip) │                   │
│  ├──────────────────────┼──────────────────────┤                   │
│  │ Activity Selection   │  Weighted Job Sched. │                   │
│  │ → GREEDY (max count) │  → DP (max profit)   │                   │
│  ├──────────────────────┼──────────────────────┤                   │
│  │ Jump Game I (bool)   │  Coin Change (min #) │                   │
│  │ → GREEDY (reachable?)│  → DP (optimal combo)│                   │
│  └──────────────────────┴──────────────────────┘                   │
└────────────────────────────────────────────────────────────────────┘
```

---

## Practice Problems

| # | Problem | LC # | Pattern | Difficulty |
|---|---------|------|---------|------------|
| 1 | Jump Game I | 55 | Farthest reachable | Medium |
| 2 | Jump Game II | 45 | BFS-level greedy | Medium |
| 3 | Gas Station | 134 | Running surplus | Medium |
| 4 | Candy | 135 | Two-pass L→R, R←L | Hard |
| 5 | Non-overlapping Intervals | 435 | Sort by end, keep greedily | Medium |
| 6 | Assign Cookies | 455 | Sort + two pointers | Easy |
| 7 | Task Scheduler | 621 | Frame from max freq | Medium |
| 8 | Minimum Number of Arrows to Burst Balloons | 452 | Sort by end, sweep | Medium |
| 9 | Queue Reconstruction by Height | 406 | Sort + insert | Medium |
| 10 | Partition Labels | 763 | Track last occurrence | Medium |
| 11 | Reorganize String | 767 | Max heap + cooldown | Medium |
| 12 | Boats to Save People | 881 | Sort + two pointers | Medium |
| 13 | Minimum Platforms (GFG) | — | Event sweep | Medium |
| 14 | Lemonade Change | 860 | Greedy change-making | Easy |

---

## Greedy Pattern Cheat Sheet

```
╔══════════════════════════════════════════════════════════════════════╗
║              GREEDY PATTERN CHEAT SHEET                            ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  "Maximum non-overlapping"     → Sort by end time, pick greedily     ║
║  "Minimum removals/overlaps"   → Same as above, count skipped        ║
║  "Can I reach the end?"        → Track farthest reachable            ║
║  "Minimum jumps/stops"         → BFS-style greedy levels             ║
║  "Circular route"              → Track running surplus               ║
║  "Distribute fairly"           → Two-pass: left-to-right, R←L       ║
║  "Minimum rooms/platforms"     → Sort events, sweep with counter     ║
║  "Match items to slots"        → Sort both, greedy two-pointer       ║
║  "Minimize idle/wasted time"   → Frame from most frequent task       ║
║                                                                      ║
║  THE GREEDY PROOF TEMPLATE:                                          ║
║  1. Assume an optimal solution OPT                                   ║
║  2. Show you can swap OPT's choice with GREEDY's choice              ║
║  3. Show the swap doesn't make it worse                              ║
║  4. Repeat → GREEDY = OPT  ∎                                        ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
```

---

```
╔══════════════════════════════════════════════════════════════════════╗
║                                                                      ║
║     🎉  Congratulations — you've completed the core DSA roadmap!     ║
║                                                                      ║
║     Topics 0-12 covered:                                             ║
║       Python foundations, complexity, arrays, linked lists,           ║
║       stacks & queues, hash maps, trees, graphs, heaps,              ║
║       recursion & backtracking, sorting, dynamic programming,        ║
║       and greedy algorithms.                                         ║
║                                                                      ║
║     You have the toolkit. Now go solve problems.                     ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
```